# GeoFire-X — Ablation indépendante de pos_weight
Poids préenregistrés : **100 (contrôle), 30, 10**, pour CNN B1 et U-Net B2.
Chaque variante initialise un nouveau modèle, seed 42, sans charger les checkpoints
historiques. Budget fixe : 4 époques / 23 404 updates, batch train 16, AdamW identique,
Dice weight 0,5, corpus et normalisation inchangés. Les données déjà calibrées ne
sont pas recalibrées par ce notebook : seuls les nouveaux modèles seront évalués.

Attacher les **11 partitions compactes existantes**. Aucun dataset ni checkpoint
supplémentaire à transférer. GPU T4, Internet désactivé. L'entraînement reste désactivé
par défaut ; ne l'activer qu'après autorisation du budget. Une seule variante par
lancement. La limite de temps sauvegarde un checkpoint de reprise dans son nouveau run.


In [ ]:
import base64, hashlib, io, json, os, sys, tempfile, time, zipfile, subprocess
from pathlib import Path
workspace=Path(tempfile.mkdtemp(prefix="geofirex-loss-code-",dir="/tmp"))
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='603b4c28908af47a069c57b235a0392a6a2a3d5ee7a20344eece6ddf75c80a7e'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    z.extractall(workspace)
sys.path.insert(0,str(workspace))
os.environ["PYTHONPATH"]=str(workspace)+os.pathsep+os.environ.get("PYTHONPATH", "")
from geofirex.calibration_mounts import restore_verified
expected=['ed88f93ae8329c57f34709fdf3a59e3ec3272db8238f843d4329f2cb23e293d0', '4aa8b2818192a2cc67941f1c1bbb8a1001e144114486bb05ecaa9b504993c6fa', 'b5caf48618be26a75a480d91401f9de0d0936607936d09556f6308c1d76c5dcd', 'ba72fbbc58f398a7bcd471d5cb21aebe4d11ed68da02caa912c2e746ccadc04d', 'ada2f8b9a3f7f340efed521b15bf3fe4e3b4151fa39eab06ca2055518f58ab58', '3bcde37b262d5828f98153350d961e0ad771032fdedb04ff180ace536e4cb33a', '001a171f309ac58f4160269d46b9163b0c0c26c7a488cc545f6fa1b42e55d72a', '906b02993f756f4b6098ac8da11ecac173f59d20ebda70c8d2bd3bb37b8f6dbd', '793223fda8a5f0d06e792bcd09b2604a97252952a5323b144184a8e4f1496a4f', 'af3015768e0174071c4d197ad31681ef20fbe0cc1de2c1084254bfef5b91393f', '49e5551860a6d14c020c86dc33118a7cd2e0ca3dc9b470e6e95af6061817fa81']
maps=[p for p in Path("/kaggle/input").rglob("mount-map.json") if json.loads(p.read_text())["identity"] in expected]
corpus=restore_verified(maps,expected,Path(tempfile.mkdtemp(prefix="geofirex-loss-mount-",dir="/tmp"))/"corpus")
index=corpus/"index"


In [ ]:
# Précontrôle uniquement : aucune tuile n'est entraînée ou inférée ici.
MODEL, POS_WEIGHT = "B1", 100
OUT=Path("/kaggle/working")/(f"{MODEL}-w{POS_WEIGHT}-independent-"+time.strftime("%Y%m%d_%H%M%S"))
command=[sys.executable,"-m","geofirex.loss_ablation","--corpus",str(corpus),"--index",str(index),
         "--protocol",str(workspace/"protocol.json"),"--model",MODEL,"--pos-weight",str(POS_WEIGHT),
         "--out",str(OUT),"--device","cuda","--workers","4","--seconds","5400"]
subprocess.run(command,check=True,env=os.environ.copy())


## Lancement après autorisation seulement
Six exécutions indépendantes à prévoir : (B1,100), (B1,30), (B1,10), (B2,100),
(B2,30), (B2,10). La commande ci-dessous ne lance qu'un couple. Même état initial
par architecture : contrôler `initial_state_sha256` dans les six `config.json`.
La durée de 5 400 s est une limite par session, pas une estimation de quatre époques.
En cas de pause, sauvegarder les outputs, définir RESUME sur le **nouveau run de
cette même variante**, garder modèle/poids identiques et choisir un nouveau OUT.
Ne jamais utiliser les checkpoints B1/B2 historiques comme RESUME.


In [ ]:
RUN_TRAINING=False  # Activer seulement après autorisation de la campagne/budget.
RESUME=None
if RUN_TRAINING:
    training_command=command+["--execute-training"]
    if RESUME: training_command += ["--resume",str(RESUME)]
    subprocess.run(training_command,check=True,env=os.environ.copy())
else:
    print("Entraînement désactivé. Précontrôle terminé.")


## Sélection finale sur les 32 seulement
Le checkpoint comparé est fixé à l'époque 4 pour toutes les variantes. La grille
de seuils est préenregistrée : 0,10–0,95 par 0,005, puis 0,96 / 0,97 / 0,98 / 0,99 /
0,995 / 0,999. AP et Brier utilisent les probabilités. Aucun pixel des 169 événements
ou du test 2021 n'est inféré. Après les six runs, utiliser `select_loss_ablation.py`
avec leurs six dossiers : Dice micro, égalité AP puis Brier. Figer le gagnant et son
seuil avant une éventuelle confirmation unique ; les 169 ne serviront jamais à
relancer la recherche. Aucun entraînement ou résultat nouveau n'est revendiqué ici.
